In [ ]:
import os

os.chdir('./drive/MyDrive/POGNN_Module')

In [ ]:
! ls

batch_effects.ipynb
build_dummy_data.ipynb
class_results.pt
content
Dataset
dummy_data
gene_network.gexf
Model
model_epoch200_default_Cos5_Complete.pth
model_epoch200_default_CosFace.pth
model_epoch200_GATSoftMaxP5k_CosFace.pth
model_epoch200_Prune5k_CosFace.pth
model_epoch20_default.pth
model_epoch400_GATSoftMaxP5k_CosFace.pth
network_matrix.html
outputs
requirements.txt
runner.ipynb
runs
tea_debug.log
tester.ipynb
val_class_results_GATSoftMaxP5k.pth
valid_result_GATSoftMaxP5k.pth
visualization
wandb


In [ ]:
! pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 12.1 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement pyg_lib==0.6.0+pt28cpu (from versions: none)
ERROR: No matching distribution found for pyg_lib==0.6.0+pt28cpu


In [ ]:
! pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [ ]:
! pip install torch_geometric biopython mygene neo4j harmonypy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
  Using cached biopython-1.87-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached mygene-3.2.2-py2.py3-none-any.whl.metadata (10 kB)
  Using cached biothings_client-0.5.0-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00


In [ ]:
from torch_geometric.loader import DataLoader
from Dataset.TemplateBuild import GraphDataset, GraphSnapshot, DatasetBuilder
from Dataset.utils import GraphTraverse
from visualization.chord_diagram import global_layout


graph_snapshot = GraphSnapshot('./Dataset/result/ML_Dataset/graph_snapshot.pkl')
ds = DatasetBuilder('./Dataset/result/', graph_snapshot=graph_snapshot)
gt = GraphTraverse(graph_snapshot)
graph = ds.gs.graph
gl_layout = global_layout(graph, gt)

train_dataset = GraphDataset(root='./Dataset/result/ML_Dataset', split='train')
test_dataset = GraphDataset(root='./Dataset/result/ML_Dataset', split='test')
val_dataset = GraphDataset(root='./Dataset/result/ML_Dataset', split='valid')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Loading graph from ./Dataset/result/ML_Dataset/graph_snapshot.pkl...


In [ ]:
import json

with open('./Model/tissue_classes.json') as f:
  label_map = json.load(f)
  label_map = {int(k):v for k,v in label_map.items()}

print(type(label_map), label_map)

<class 'dict'> {0: 'Adipose Subcutaneous', 1: 'Artery Tibial', 2: 'Breast Mammary Tissue', 3: 'Cells Cultured Fibroblasts', 4: 'Esophagus Mucosa', 5: 'Lung', 6: 'Muscle Skeletal', 7: 'Nerve Tibial', 8: 'Thyroid', 9: 'Whole Blood'}


In [ ]:
import torch
import importlib
import Model.model3
importlib.reload(Model.model3)
from Model.model3 import TissueClassificationPipeline_Model3

import Model.layers3.proto_mem
importlib.reload(Model.layers3.proto_mem)
from Model.layers3.proto_mem import Prototype_Memory



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


GAT_HID_DIM = 64
NORM_HID_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
TEMPERATURE = 0.07
MARGIN = 0.35
BACKBONE_LR = 5e-3
HEAD_LR = 1e-4

proto_mem = Prototype_Memory(class_num= len(label_map), device=device)

model = TissueClassificationPipeline_Model3(
    label_map = label_map,
    memory = proto_mem,
    in_channels = 1,
    norm_hidden = NORM_HID_DIM,
    gat_hidden = GAT_HID_DIM,
    gat_heads = NUM_HEADS,
    dropout = DROPOUT,
    margin = MARGIN,
    temperature = TEMPERATURE,
)



model = model.to(device)
optimizer = torch.optim.AdamW(model.parameter_groups(backbone_lr=BACKBONE_LR, head_lr=HEAD_LR), weight_decay=1e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('wandb_key'))
wandb.finish()

total_epochs = 60
warmup_epochs = 15

run = wandb.init(
    entity="pathwayoracle",
    project="GATModel3_50E_Transl_AllDataset",
    resume=False,
    # id="325e5udi",
    config={
        "learning_rate": BACKBONE_LR,
        "architecture": "GNN",
        "dataset": "real-dataset-normal-tissue-n10",
        "epochs": total_epochs,
        "gat_hidden_dim": GAT_HID_DIM,
        "norm_hidden_dim": NORM_HID_DIM,
        "num_heads": NUM_HEADS,
        "dropout": DROPOUT,
        "margin": MARGIN,
        "temperature": TEMPERATURE,
    },
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
import importlib

import torch
import wandb

import visualization.umap
import Model.trainer
import Model.evaluater
import visualization.umap
import visualization.visualizer

importlib.reload(Model.trainer)
importlib.reload(Model.evaluater)
importlib.reload(visualization.umap)
importlib.reload(visualization.visualizer)

from Model.trainer import Trainer
from Model.evaluater import Evaluater
from Model.schedulers import LinearWarmup, CosineDescent
from visualization.umap import UMAPTransform
from visualization.plot_correlation import ConsistencyTracker
from visualization.visualizer import Visualizer


warmup = LinearWarmup(optimizer, warmup_epochs=warmup_epochs)
cosine = CosineDescent(optimizer, window=5)


umap_model = UMAPTransform(model)
umap_translator = UMAPTransform(model)

visualizer = Visualizer(run, graph, gt, umap_model, umap_translator)
train_tracker = ConsistencyTracker(model.label_map, split='train')
val_tracker = ConsistencyTracker(model.label_map, split='val')


trainer = Trainer(model=model,
                  optimizer=optimizer,
                  device=device,
                  run=run,
                  visualizer=visualizer,
                  consistency_tracker = train_tracker)

evaluater = Evaluater(model=model,
                      optimizer=optimizer,
                      device=device,
                      run=run,
                      visualizer=visualizer,
                      consistency_tracker= val_tracker)


In [ ]:
import time

for epoch in range(1, total_epochs + 1):
    start_time = time.time()

    tr_model_results, tr_translator_results = trainer.train(train_loader, epoch)

    if epoch < warmup_epochs:
        warmup.step()
    else:
        cosine.step(tr_model_results.loss)

    run.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    if epoch % 5 == 0:
        tr_class_results = trainer.edge_mask_explain(tr_model_results, train_loader)
        trainer.visualize_all(epoch, tr_model_results, tr_translator_results, tr_class_results)

    if epoch % 20 == 0:
        val_model_results, val_translator_results = evaluater.evaluate(val_loader, epoch)
        val_class_results = evaluater.edge_mask_explain(val_model_results, val_loader)
        evaluater.visualize_all(epoch, val_model_results, val_translator_results, val_class_results)

    print(f"--- {time.time() - start_time:.2f} seconds ---")


  Epoch 1  model_loss=24.8708
--- 26.58 seconds ---
  Epoch 2  model_loss=21.1660
--- 27.01 seconds ---
  Epoch 3  model_loss=20.3327
--- 26.67 seconds ---
  Epoch 4  model_loss=20.4311
--- 26.75 seconds ---
  Epoch 5  model_loss=19.7478
--- 150.03 seconds ---
  Epoch 6  model_loss=19.4627
--- 26.89 seconds ---
  Epoch 7  model_loss=19.0042
--- 26.90 seconds ---
  Epoch 8  model_loss=19.9382
--- 26.94 seconds ---
  Epoch 9  model_loss=19.5739
--- 27.00 seconds ---
  Epoch 10  model_loss=18.9626
--- 149.40 seconds ---
  Epoch 11  model_loss=18.0981
--- 26.75 seconds ---
  Epoch 12  model_loss=17.9231
--- 26.76 seconds ---
  Epoch 13  model_loss=17.8709
--- 26.78 seconds ---
  Epoch 14  model_loss=16.9088
--- 27.74 seconds ---
  Epoch 15  model_loss=17.3857
--- 151.56 seconds ---
  Epoch 16  model_loss=16.8036
--- 26.99 seconds ---
  Epoch 17  model_loss=16.2972
--- 26.85 seconds ---
  Epoch 18  model_loss=16.3426
--- 26.80 seconds ---
  Epoch 19  model_loss=14.8730
--- 26.81 seconds ---

In [ ]:
for epoch in range(61, 200 + 1):
    start_time = time.time()

    tr_model_results, tr_translator_results = trainer.train(train_loader, epoch)

    if epoch < warmup_epochs:
        warmup.step()
    else:
        cosine.step(tr_model_results.loss)

    run.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    if epoch % 5 == 0:
        tr_class_results = trainer.edge_mask_explain(tr_model_results, train_loader)
        trainer.visualize_all(epoch, tr_model_results, tr_translator_results, tr_class_results)

    if epoch % 20 == 0:
        val_model_results, val_translator_results = evaluater.evaluate(val_loader, epoch)
        val_class_results = evaluater.edge_mask_explain(val_model_results, val_loader)
        evaluater.visualize_all(epoch, val_model_results, val_translator_results, val_class_results)

    print(f"--- {time.time() - start_time:.2f} seconds ---")

  Epoch 61  model_loss=14.3090
--- 30.76 seconds ---
  Epoch 62  model_loss=14.5069
--- 30.96 seconds ---
  Epoch 63  model_loss=14.2608
--- 30.82 seconds ---
  Epoch 64  model_loss=14.2320
--- 30.81 seconds ---
  Epoch 65  model_loss=14.2077
--- 161.28 seconds ---
  Epoch 66  model_loss=14.0919
--- 30.60 seconds ---
  Epoch 67  model_loss=14.0805
--- 30.87 seconds ---
  Epoch 68  model_loss=13.9687
--- 30.67 seconds ---
  Epoch 69  model_loss=13.7239
--- 30.86 seconds ---
0.06725116101648734
  Epoch 70  model_loss=13.4617
--- 204.41 seconds ---
  Epoch 71  model_loss=13.6522
--- 32.21 seconds ---
  Epoch 72  model_loss=13.5240
--- 31.85 seconds ---
  Epoch 73  model_loss=13.2540
--- 31.99 seconds ---
  Epoch 74  model_loss=13.1547
--- 31.68 seconds ---
  Epoch 75  model_loss=13.2194
--- 174.35 seconds ---
  Epoch 76  model_loss=13.0909
--- 31.53 seconds ---
  Epoch 77  model_loss=12.9038
--- 31.70 seconds ---
  Epoch 78  model_loss=12.8385
--- 31.58 seconds ---
  Epoch 79  model_loss=

In [ ]:
for epoch in range(201, 300 + 1):
    start_time = time.time()

    tr_model_results, tr_translator_results = trainer.train(train_loader, epoch)

    if epoch < warmup_epochs:
        warmup.step()
    else:
        cosine.step(tr_model_results.loss)

    run.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    if epoch % 5 == 0:
        tr_class_results = trainer.edge_mask_explain(tr_model_results, train_loader)
        trainer.visualize_all(epoch, tr_model_results, tr_translator_results, tr_class_results)

    if epoch % 20 == 0:
        val_model_results, val_translator_results = evaluater.evaluate(val_loader, epoch)
        val_class_results = evaluater.edge_mask_explain(val_model_results, val_loader)
        evaluater.visualize_all(epoch, val_model_results, val_translator_results, val_class_results)

    print(f"--- {time.time() - start_time:.2f} seconds ---")

  Epoch 201  model_loss=5.5510
--- 31.54 seconds ---
  Epoch 202  model_loss=9.9577
--- 31.63 seconds ---
  Epoch 203  model_loss=9.6565
--- 31.15 seconds ---
  Epoch 204  model_loss=8.7380
--- 31.12 seconds ---
  Epoch 205  model_loss=8.0461
--- 175.68 seconds ---
  Epoch 206  model_loss=7.7108
--- 30.93 seconds ---
  Epoch 207  model_loss=6.7916
--- 31.08 seconds ---
  Epoch 208  model_loss=6.6613
--- 31.08 seconds ---
  Epoch 209  model_loss=6.4418
--- 31.05 seconds ---
-0.024343517219916174
  Epoch 210  model_loss=6.1208
--- 175.33 seconds ---
  Epoch 211  model_loss=5.8303
--- 30.91 seconds ---
  Epoch 212  model_loss=5.8976
--- 31.04 seconds ---
  Epoch 213  model_loss=5.7989
--- 31.05 seconds ---
  Epoch 214  model_loss=5.6172
--- 31.06 seconds ---
  Epoch 215  model_loss=5.3690
--- 177.57 seconds ---
  Epoch 216  model_loss=5.6701
--- 30.98 seconds ---
  Epoch 217  model_loss=5.2566
--- 31.08 seconds ---
  Epoch 218  model_loss=5.1590
--- 30.99 seconds ---
  Epoch 219  model_lo

In [ ]:
for epoch in range(301, 360 + 1):
    start_time = time.time()

    tr_model_results, tr_translator_results = trainer.train(train_loader, epoch)

    if epoch < warmup_epochs:
        warmup.step()
    else:
        cosine.step(tr_model_results.loss)

    run.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    if epoch % 5 == 0:
        tr_class_results = trainer.edge_mask_explain(tr_model_results, train_loader)
        trainer.visualize_all(epoch, tr_model_results, tr_translator_results, tr_class_results)

    if epoch % 20 == 0:
        val_model_results, val_translator_results = evaluater.evaluate(val_loader, epoch)
        val_class_results = evaluater.edge_mask_explain(val_model_results, val_loader)
        evaluater.visualize_all(epoch, val_model_results, val_translator_results, val_class_results)

    print(f"--- {time.time() - start_time:.2f} seconds ---")

  Epoch 301  model_loss=4.1882
--- 31.82 seconds ---
  Epoch 302  model_loss=3.9231
--- 31.91 seconds ---
  Epoch 303  model_loss=3.7859
--- 31.98 seconds ---
  Epoch 304  model_loss=3.8460
--- 31.87 seconds ---
  Epoch 305  model_loss=3.9862
--- 180.67 seconds ---
  Epoch 306  model_loss=3.6990
--- 32.18 seconds ---
  Epoch 307  model_loss=3.6710
--- 32.33 seconds ---
  Epoch 308  model_loss=3.6681
--- 32.44 seconds ---
  Epoch 309  model_loss=3.5877
--- 32.22 seconds ---
0.08445973754145011
  Epoch 310  model_loss=3.7984
--- 181.82 seconds ---
  Epoch 311  model_loss=3.8908
--- 31.98 seconds ---
  Epoch 312  model_loss=3.6799
--- 32.28 seconds ---
  Epoch 313  model_loss=3.7714
--- 32.06 seconds ---
  Epoch 314  model_loss=3.7655
--- 32.27 seconds ---
  Epoch 315  model_loss=5.3855
--- 176.20 seconds ---
  Epoch 316  model_loss=5.2603
--- 32.06 seconds ---
  Epoch 317  model_loss=5.2546
--- 32.20 seconds ---
  Epoch 318  model_loss=4.8700
--- 32.07 seconds ---
  Epoch 319  model_loss

In [ ]:
model.warmup_complete = True

In [ ]:
wandb.finish()


inter_cos_similarity/train/Adipose Subcutaneous,▁▆▆▁▆▇▇▇▇▇██▇█▇████████▇████████████████
inter_cos_similarity/train/Artery Tibial,▄▂▄▆▅▁██▇▇████▇▇▇██▇█▇▇▇▇█▇▇████▇▇██▇▇▇▇
inter_cos_similarity/train/Breast Mammary Tissue,▃▆▁▁▃▅▅▇▅▅▇▇▆▇▆▆██▆▇█▇▇▆▅▆▅▆
inter_cos_similarity/train/Cells Cultured Fibroblasts,▇▁▆▆▆▇█▇██████▇▇█▇███▇▇▆▇▇▇▇▇█▇▇▇▇██▇█▇▆
inter_cos_similarity/train/Esophagus Mucosa,▁▃▆▆▇▇██▇███▇█████████████████▇█████▇██▇
inter_cos_similarity/train/Lung,▁▅▆▆▆▇████▇█▇▇██████▇▇▇████▆▇▇▇▇▇█▇████▇
inter_cos_similarity/train/Muscle Skeletal,▇▆▁▂▄▇▇████▇▇█████▇█▇▇▇█████████████████
inter_cos_similarity/train/Nerve Tibial,▆▅▁▆▇▇▇▇▇▇██▇▇█████████████████▇███████▇
inter_cos_similarity/train/Thyroid,▂▄▁▃▄▄▄▅▇▇▇▇▇▇██▇█▇▇████████████████████
inter_cos_similarity/train/Whole Blood,▇▇▁▆▆▇█▇▇███████▇███▆▇█▇█████▇█████▇▇▇██
+39,...


In [ ]:
len(model.proto_mem.prototypes)

334

In [ ]:
type(model.proto_mem.prototypes)

list

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import numpy as np

for class_label, queue in model.proto_mem.memory_queue.items():
    embeddings_tensor = torch.stack(list(queue))

    clustering = AgglomerativeClustering(
        metric='cosine',
        linkage='average',
        distance_threshold=None,
        n_clusters=1,
        compute_full_tree=True,
        compute_distances=True
    )
    clustering.fit(embeddings_tensor.numpy())

    gaps = np.diff(clustering.distances_)
    # Merge distance right after the biggest gap.
    discovered_threshold = clustering.distances_[gaps.argmax() + 1]

    cluster_labels = AgglomerativeClustering(
        metric='cosine',
        linkage='average',
        distance_threshold=discovered_threshold,
        n_clusters=None,
        compute_full_tree=True
    ).fit_predict(embeddings_tensor.numpy())

    n_clusters = cluster_labels.max() + 1
    print(f"class={class_label} → threshold={discovered_threshold:.4f} → {n_clusters} clusters")

class=0 → threshold=0.7202 → 2 clusters
class=1 → threshold=1.0382 → 2 clusters
class=2 → threshold=0.7733 → 4 clusters
class=3 → threshold=0.7978 → 2 clusters
class=4 → threshold=0.6884 → 2 clusters
class=5 → threshold=0.7342 → 4 clusters
class=6 → threshold=0.6761 → 2 clusters
class=7 → threshold=0.4982 → 2 clusters
class=8 → threshold=1.2144 → 2 clusters
class=9 → threshold=0.1266 → 2 clusters


In [ ]:
torch.save({
        "model_state": model.state_dict(),
        "proto_mem": model.proto_mem.memory_queue,
    }, f"{run.project}.pt")

In [ ]:
def results_save(model_results, translator_results, path):
  torch.save({
      "model_results": model_results,
      "translator_results": translator_results,
  }, path)

results_save(tr_model_results, tr_translator_results, f"TRGATResults_{run.project}.pth")

results_save(val_model_results, val_translator_results, f"VALGATResults_{run.project}.pth")

In [ ]:
import dataclasses
from collections import deque

def _make_picklable(obj):
    """Recursively convert dataclass instances → dicts, deques → lists."""
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return {f.name: _make_picklable(getattr(obj, f.name))
                for f in dataclasses.fields(obj)}
    elif isinstance(obj, dict):
        return {k: _make_picklable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, deque)):
        return [_make_picklable(x) for x in obj]
    elif isinstance(obj, tuple):
        return tuple(_make_picklable(x) for x in obj)
    return obj   # tensors, scalars, strings, etc. pass through

def model_save(model, path: str):
    torch.save({
        "model_state": model.state_dict(),
        "proto_mem": _make_picklable(model.proto_mem),
    }, path)

def results_save(model_results, translator_results, path):
  torch.save({
      "model_results": model_results,
      "translator_results": translator_results,
  }, path)

model_save(model, f"{run.project}.pt")
results_save(val_model_results, val_translator_results, f"GATResults_{run.project}.pth")

PicklingError: Can't pickle <class 'Model.data_model.PrototypeEmbedding'>: it's not the same object as Model.data_model.PrototypeEmbedding

###Test 2- Incremental additions of classes. Start with 5 classes, then add, 1, 3, 1; to test if big increments / small increments effect loss.

In [ ]:
import visualization.confusion_matrix

importlib.reload(visualization.confusion_matrix )
from visualization.confusion_matrix import plot_confusion_with_confidence


run = wandb.init(
    entity="pathwayoracle",
    project="GATModel3_50E_Transl_IncrementalDataset",
    resume=False,
    # id="325e5udi",
    config={
        "learning_rate": BACKBONE_LR,
        "architecture": "GNN",
        "dataset": "real-dataset-normal-tissue-n10",
        "epochs": total_epochs,
        "gat_hidden_dim": GAT_HID_DIM,
        "norm_hidden_dim": NORM_HID_DIM,
        "num_heads": NUM_HEADS,
        "dropout": DROPOUT,
        "margin": MARGIN,
        "temperature": TEMPERATURE,
    },
)

In [ ]:
import json

with open('./Model/tissue_classes.json') as f:
  json_map = json.load(f)
  label_map = {int(k):v for i, (k,v) in enumerate(json_map.items()) if i <5}
  incremental_map = {int(k):v for i, (k,v) in enumerate(json_map.items()) if i >=5}

print(type(label_map), label_map)
print(type(incremental_map), incremental_map)

<class 'dict'> {0: 'Adipose Subcutaneous', 1: 'Artery Tibial', 2: 'Breast Mammary Tissue', 3: 'Cells Cultured Fibroblasts', 4: 'Esophagus Mucosa'}
<class 'dict'> {5: 'Lung', 6: 'Muscle Skeletal', 7: 'Nerve Tibial', 8: 'Thyroid', 9: 'Whole Blood'}


In [ ]:
import torch
import importlib
import Model.model3
importlib.reload(Model.model3)
from Model.model3 import TissueClassificationPipeline_Model3



GAT_HID_DIM = 64
NORM_HID_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
TEMPERATURE = 0.07
TOPK_EDGES = 5000
MARGIN = 0.35
BACKBONE_LR = 5e-3
HEAD_LR = 1e-4

model = TissueClassificationPipeline_Model3(
    label_map = label_map,
    in_channels = 1,
    norm_hidden = NORM_HID_DIM,
    gat_hidden = GAT_HID_DIM,
    gat_heads = NUM_HEADS,
    dropout = DROPOUT,
    topk_edges = TOPK_EDGES,
    margin = MARGIN,
    temperature = TEMPERATURE,
)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameter_groups(backbone_lr=BACKBONE_LR, head_lr=HEAD_LR), weight_decay=1e-5)

In [ ]:
import importlib

import torch
import wandb

import visualization.umap

importlib.reload(Model.trainer)
importlib.reload(Model.evaluater)
importlib.reload(visualization.umap)

from Model.trainer import Trainer
from Model.evaluater import Evaluater
from Model.schedulers import LinearWarmup, CosineDescent
from visualization.umap import UMAPTransform
from visualization.plot_correlation import ConsistencyTracker
from visualization.visualizer import Visualizer


warmup = LinearWarmup(optimizer, warmup_epochs=warmup_epochs)
cosine = CosineDescent(optimizer, window=5)


umap_model = UMAPTransform(model)
umap_translator = UMAPTransform(model)

visualizer = Visualizer(run, graph, gt, umap_model, umap_translator)
train_tracker = ConsistencyTracker(model.label_map, split='train')
val_tracker = ConsistencyTracker(model.label_map, split='val')


trainer = Trainer(model=model,
                  optimizer=optimizer,
                  device=device,
                  run=run,
                  visualizer=visualizer,
                  consistency_tracker = train_tracker)

evaluater = Evaluater(model=model,
                      optimizer=optimizer,
                      device=device,
                      run=run,
                      visualizer=visualizer,
                      consistency_tracker= val_tracker)


In [ ]:
increments = [1, 3, 1]
i = 0
v = 0
for epoch in range(1, total_epochs + 1):
    tr_model_results, tr_translator_results = trainer.train(train_loader, epoch)

    if epoch < warmup_epochs:
        warmup.step()
    else:
        cosine.step(tr_model_results.loss)

    run.log({'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    if epoch % 5 == 0:
        tr_class_results = trainer.edge_mask_explain(tr_model_results, train_loader)
        trainer.visualize_all(epoch, tr_model_results, tr_translator_results, tr_class_results)

    if epoch % 20 == 0:
        val_model_results, val_translator_results = evaluater.evaluate(val_loader, epoch)
        val_class_results = evaluater.edge_mask_explain(val_model_results, val_loader)
        evaluater.visualize_all(epoch, val_model_results, val_translator_results, val_class_results)

        increment_keys = list(incremental_map.keys())[v: v+ increments[i]]
        v = i + increments[i]
        i += 1
        trainer.sync_classes_from(label_map={k: v for k, v in incremental_map.items() if k in increment_keys})

In [ ]:
wandb.finish()

In [ ]:
def model_save(model, path: str):
  torch.save({
      "model_state": model.state_dict(),
      "proto_mem": model.proto_mem,
  }, path)

def results_save(model_results, translator_results, path):
  torch.save({
      "model_results": model_results,
      "translator_results": translator_results,
  }, path)

model_save(model, f"{run.project}.pt")
results_save(val_model_results, val_translator_results, f"GATResults_{run.project}.pth")